<a href="https://colab.research.google.com/github/427paul/Machine_Learning/blob/main/DACON_x_BDA_%ED%95%99%EC%8A%B5%EC%9E%90_%EC%88%98%EB%A3%8C_%EC%98%88%EC%B8%A1_AI_%EA%B2%BD%EC%A7%84%EB%8C%80%ED%9A%8C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 0.4268

In [1]:
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder, MultiLabelBinarizer
from lightgbm import LGBMClassifier
from sklearn.metrics import f1_score

# 1. 정제용 함수 정의
def clean_gain(text):
    if any(word in str(text) for word in ['전체', '전부', '모두', '4가지', '어렵습니다']):
        return '항목 전체'
    if '공모전' in str(text) and '프로젝트' in str(text):
        return '공모전 및 프로젝트'
    return text

# 2. 통합 전처리 파이프라인 (Train/Test 한 번에 처리)
def preprocess_pipeline(train_df, test_df):
    y_train = train_df['completed']
    test_ids = test_df['ID']

    # Train과 Test를 위아래로 병합 (인코딩 불일치 완벽 방지)
    df = pd.concat([train_df.drop(columns=['completed']), test_df], axis=0).reset_index(drop=True)

    # (1) 불필요 컬럼 제거
    drop_cols = ['ID', 'generation', 'contest_award', 'idea_contest', 'contest_participation',
                 'previous_class_3', 'previous_class_4', 'previous_class_5', 'previous_class_6',
                 'previous_class_7', 'previous_class_8', 'incumbents_lecture_scale_reason']
    df = df.drop(columns=[c for c in drop_cols if c in df.columns], errors='ignore')

    # (2) 수치형 데이터 처리
    if 'completed_semester' in df.columns:
        df['completed_semester'] = df['completed_semester'].fillna(df['completed_semester'].median())

    for col in ['class2', 'class3', 'class4']:
        if col in df.columns:
            df[col] = df[col].fillna(0)

    # (3) 텍스트 데이터 정제
    if 'what_to_gain' in df.columns:
        df['what_to_gain'] = df['what_to_gain'].apply(clean_gain)

    # (4) 멀티-레이블 전처리 (원-핫 인코딩)
    multilabel_cols = ['major_field', 'desired_job', 'certificate_acquisition', 'desired_certificate',
                       'desired_job_except_data', 'interested_company', 'expected_domain', 'onedayclass_topic']

    for col in multilabel_cols:
        if col in df.columns:
            df[col] = df[col].fillna('')
            s_list = df[col].apply(lambda x: [i.strip() for i in str(x).split(',')] if x != '' else [])
            mlb = MultiLabelBinarizer()
            mlb_result = mlb.fit_transform(s_list)
            new_cols = [f"{col}_{str(cls).replace(' ', '_').replace(',', '')}" for cls in mlb.classes_]
            mlb_df = pd.DataFrame(mlb_result, columns=new_cols, index=df.index)
            df = pd.concat([df.drop(columns=[col]), mlb_df], axis=1)

    # (5) 남은 범주형 변수 라벨 인코딩
    cat_cols = df.select_dtypes(include=['object', 'bool']).columns
    for col in cat_cols:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].astype(str))

    # (6) 데이터 다시 분리
    n_train = len(train_df)
    X_train = df.iloc[:n_train].copy()
    X_test = df.iloc[n_train:].copy()

    return X_train, y_train, X_test, test_ids

# --- 메인 실행부 ---

# 1. 데이터 로드
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

# 2. 전처리 실행
X, y, X_test, test_ids = preprocess_pipeline(train_df, test_df)

# 3. 컬럼명 특수문자 정제 및 중복 제거 (LightGBM 에러 방지)
raw_cols = [re.sub(r'[^\w]+', '_', col) for col in X.columns]
final_cols = []
seen = {}
for col in raw_cols:
    if col in seen:
        seen[col] += 1
        final_cols.append(f"{col}{seen[col]}")
    else:
        seen[col] = 0
        final_cols.append(col)
X.columns = final_cols
X_test.columns = final_cols

# 4. 5-Fold 교차 검증 및 모델 학습
n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

oof_preds_proba = np.zeros(len(X))
test_preds_proba = np.zeros(len(X_test))
models = []

print("=== 5-Fold LightGBM 학습 시작 ===")
for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
    X_va, y_va = X.iloc[val_idx], y.iloc[val_idx]

    model = LGBMClassifier(
        n_estimators=1000,     # 트리 개수를 넉넉히 주고
        learning_rate=0.03,    # 학습률을 낮춰 세밀하게 학습
        max_depth=8,
        num_leaves=63,
        subsample=0.8,
        colsample_bytree=0.8,
        class_weight='balanced',
        random_state=42,
        verbosity=-1
    )

    # 조기 종료(Early Stopping)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        callbacks=[pd.core.common.require_public_api()] if False else None
    )

    # 검증 셋 확률 예측
    val_proba = model.predict_proba(X_va)[:, 1]
    oof_preds_proba[val_idx] = val_proba

    # 테스트 셋 확률 예측 (5개 모델의 결과를 누적)
    test_preds_proba += model.predict_proba(X_test)[:, 1] / n_splits
    models.append(model)
    print(f"Fold {fold+1} 학습 완료")

# 5. OOF 예측 기반 최적 임계값(Threshold) 탐색
best_thresh = 0.5
best_f1 = 0
for t in np.arange(0.1, 0.9, 0.01):
    f1 = f1_score(y, (oof_preds_proba >= t).astype(int))
    if f1 > best_f1:
        best_f1 = f1
        best_thresh = t

print(f"\n[최종 결과]")
print(f"최적 임계값: {best_thresh:.2f}, 전체 검증 F1 Score: {best_f1:.4f}")

# 6. Test 데이터 최종 예측
final_test_preds = (test_preds_proba >= best_thresh).astype(int)

# 7. 결과 저장
submission = pd.DataFrame({
    'ID': test_ids,
    'completed': final_test_preds
})
submission.to_csv('submission_lgbm_cv.csv', index=False)
print("--- 'submission_lgbm_cv.csv' 파일 생성이 완료되었습니다. ---")

=== 5-Fold LightGBM 학습 시작 ===
Fold 1 학습 완료
Fold 2 학습 완료
Fold 3 학습 완료
Fold 4 학습 완료
Fold 5 학습 완료

[최종 결과]
최적 임계값: 0.14, 전체 검증 F1 Score: 0.3949
--- 'submission_lgbm_cv.csv' 파일 생성이 완료되었습니다. ---


0.4044

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MultiLabelBinarizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, classification_report

# 1. 정제용 함수 정의
def clean_gain(text):
    if any(word in str(text) for word in ['전체', '전부', '모두', '4가지', '어렵습니다']):
        return '항목 전체'
    if '공모전' in str(text) and '프로젝트' in str(text):
        return '공모전 및 프로젝트'
    return text

# 2. 통합 전처리 프로세스 함수 (Train/Test 공용)
def preprocess_pipeline(df):
    # (1) 불필요 컬럼 제거
    drop_cols = ['ID', 'generation', 'contest_award', 'idea_contest', 'contest_participation',
                 'previous_class_3', 'previous_class_4', 'previous_class_5', 'previous_class_6',
                 'previous_class_7', 'previous_class_8', 'incumbents_lecture_scale_reason']
    df = df.drop(columns=[c for c in drop_cols if c in df.columns])

    # (2) 수치형 데이터 처리
    if 'completed_semester' in df.columns:
        df['completed_semester'] = df['completed_semester'].fillna(df['completed_semester'].median())

    for col in ['class2', 'class3', 'class4']:
        if col in df.columns:
            df[col] = df[col].fillna(0)

    # (3) 텍스트 데이터 정제
    if 'what_to_gain' in df.columns:
        df['what_to_gain'] = df['what_to_gain'].apply(clean_gain)

    # (4) 멀티-레이블 전처리 (원-핫 인코딩)
    multilabel_cols = ['major_field', 'desired_job', 'certificate_acquisition', 'desired_certificate',
                       'desired_job_except_data', 'interested_company', 'expected_domain', 'onedayclass_topic']

    for col in multilabel_cols:
        if col in df.columns:
            df[col] = df[col].fillna('')
            s_list = df[col].apply(lambda x: [i.strip() for i in str(x).split(',')] if x != '' else [])
            mlb = MultiLabelBinarizer()
            mlb_result = mlb.fit_transform(s_list)
            new_cols = [f"{col}_{cls}" for cls in mlb.classes_]
            mlb_df = pd.DataFrame(mlb_result, columns=new_cols, index=df.index)
            df = pd.concat([df.drop(columns=[col]), mlb_df], axis=1)

    # (5) 남은 범주형 변수 라벨 인코딩
    cat_cols = df.select_dtypes(include=['object', 'bool']).columns
    le = LabelEncoder()
    for col in cat_cols:
        df[col] = le.fit_transform(df[col].astype(str))
    return df

# --- 메인 실행부 ---

# 1. 데이터 로드
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

# 2. 전처리 실행
processed_train = preprocess_pipeline(train_df)
processed_test = preprocess_pipeline(test_df)

# 3. 피처와 타겟 분리
X = processed_train.drop(columns=['completed']) if 'completed' in processed_train.columns else processed_train
y = train_df['completed']

# [중요] Test 데이터의 컬럼 구성을 Train과 동일하게 강제 조정
# Train에만 있는 컬럼은 0으로 채우고, Train에 없는 Test 컬럼은 제거함
processed_test = processed_test.reindex(columns=X.columns, fill_value=0)

# 4. 모델 학습 (검증셋 분리 포함)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
model = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, class_weight='balanced')
model.fit(X_train, y_train)

# 5. 최적 임계값 탐색 (Binary F1 Score 극대화)
y_proba_val = model.predict_proba(X_val)[:, 1]
best_thresh = 0.5
best_f1 = 0
for t in np.arange(0.1, 0.9, 0.01):
    f1 = f1_score(y_val, (y_proba_val >= t).astype(int))
    if f1 > best_f1:
        best_f1 = f1
        best_thresh = t

print(f"최적 임계값: {best_thresh:.2f}, 검증 F1 Score: {best_f1:.4f}")

# 6. Test 데이터 최종 예측
test_proba = model.predict_proba(processed_test)[:, 1]
test_preds = (test_proba >= best_thresh).astype(int)

# 7. 결과 저장 (ID와 completed만 포함)
submission = pd.DataFrame({
    'ID': pd.read_csv('test.csv')['ID'],
    'completed': test_preds
})

submission.to_csv('submission2.csv', index=False)
print("\n--- 'submission.csv' 파일 생성이 완료되었습니다. ---")

최적 임계값: 0.36, 검증 F1 Score: 0.4848

--- 'submission.csv' 파일 생성이 완료되었습니다. ---


# 0.3927

In [3]:
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder, MultiLabelBinarizer
from lightgbm import LGBMClassifier
from sklearn.metrics import f1_score

# 1. 정제용 함수 정의
def clean_gain(text):
    if any(word in str(text) for word in ['전체', '전부', '모두', '4가지', '어렵습니다']):
        return '항목 전체'
    if '공모전' in str(text) and '프로젝트' in str(text):
        return '공모전 및 프로젝트'
    return text

# 2. 통합 전처리 파이프라인
def preprocess_pipeline(train_df, test_df):
    y_train = train_df['completed']
    test_ids = test_df['ID']

    # Train과 Test 병합
    df = pd.concat([train_df.drop(columns=['completed']), test_df], axis=0).reset_index(drop=True)

    # (1) 불필요 컬럼 제거 (이번엔 incumbents_lecture_scale_reason 등 덜 중요한 텍스트도 과감히 제거)
    drop_cols = ['ID', 'generation', 'contest_award', 'idea_contest', 'contest_participation',
                 'previous_class_3', 'previous_class_4', 'previous_class_5', 'previous_class_6',
                 'previous_class_7', 'previous_class_8', 'incumbents_lecture_scale_reason', 'major1_2']
    df = df.drop(columns=[c for c in drop_cols if c in df.columns], errors='ignore')

    # (2) 수치형 결측치 처리 및 파생 변수 생성
    if 'completed_semester' in df.columns:
        df['completed_semester'] = df['completed_semester'].fillna(df['completed_semester'].median())

    # class1~4를 기반으로 수강 활동성(total_classes) 파생 변수 추가
    class_cols = ['class1', 'class2', 'class3', 'class4']
    for col in class_cols:
        if col in df.columns:
            df[col] = df[col].fillna(0)

    # 파생 변수: 0이 아닌 수강 이력이 몇 개인지 (활동의 적극성)
    df['total_classes'] = (df[class_cols] > 0).sum(axis=1)

    # (3) 텍스트 데이터 정제
    if 'what_to_gain' in df.columns:
        df['what_to_gain'] = df['what_to_gain'].apply(clean_gain)

    # (4) 멀티-레이블 전처리 (원-핫 인코딩)
    multilabel_cols = ['major_field', 'desired_job', 'certificate_acquisition', 'desired_certificate',
                       'desired_job_except_data', 'interested_company', 'expected_domain', 'onedayclass_topic']

    for col in multilabel_cols:
        if col in df.columns:
            df[col] = df[col].fillna('')
            s_list = df[col].apply(lambda x: [i.strip() for i in str(x).split(',')] if x != '' else [])
            mlb = MultiLabelBinarizer()
            mlb_result = mlb.fit_transform(s_list)
            new_cols = [f"{col}_{str(cls).replace(' ', '_').replace(',', '')}" for cls in mlb.classes_]
            mlb_df = pd.DataFrame(mlb_result, columns=new_cols, index=df.index)
            df = pd.concat([df.drop(columns=[col]), mlb_df], axis=1)

    # (5) 남은 범주형 변수 라벨 인코딩
    cat_cols = df.select_dtypes(include=['object', 'bool']).columns
    for col in cat_cols:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].astype(str))

    n_train = len(train_df)
    X_train = df.iloc[:n_train].copy()
    X_test = df.iloc[n_train:].copy()

    return X_train, y_train, X_test, test_ids

# --- 메인 실행부 ---

train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

X, y, X_test, test_ids = preprocess_pipeline(train_df, test_df)

# 컬럼명 정제
raw_cols = [re.sub(r'[^\w]+', '_', col) for col in X.columns]
final_cols = []
seen = {}
for col in raw_cols:
    if col in seen:
        seen[col] += 1
        final_cols.append(f"{col}{seen[col]}")
    else:
        seen[col] = 0
        final_cols.append(col)
X.columns = final_cols
X_test.columns = final_cols

n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

oof_preds_proba = np.zeros(len(X))
test_preds_proba = np.zeros(len(X_test))

print("=== 5-Fold LightGBM 학습 시작 ===")
for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
    X_va, y_va = X.iloc[val_idx], y.iloc[val_idx]

    # 하이퍼파라미터 경량화 & 밸런스 조정
    model = LGBMClassifier(
        n_estimators=300,        # 트리 개수 축소 (과적합 방지)
        learning_rate=0.05,
        max_depth=5,             # 깊이 축소
        num_leaves=15,           # 리프 노드 축소
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=2.0,    # class_weight='balanced' 대신 약간의 가중치만 부여
        random_state=42,
        verbosity=-1
    )

    model.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        callbacks=[pd.core.common.require_public_api()] if False else None
    )

    val_proba = model.predict_proba(X_va)[:, 1]
    oof_preds_proba[val_idx] = val_proba
    test_preds_proba += model.predict_proba(X_test)[:, 1] / n_splits
    print(f"Fold {fold+1} 학습 완료")

# 최적 임계값 탐색
best_thresh = 0.5
best_f1 = 0
for t in np.arange(0.1, 0.9, 0.01):
    f1 = f1_score(y, (oof_preds_proba >= t).astype(int))
    if f1 > best_f1:
        best_f1 = f1
        best_thresh = t

print(f"\n[최종 결과]")
print(f"최적 임계값: {best_thresh:.2f}, 전체 검증 F1 Score: {best_f1:.4f}")

# Test 예측
final_test_preds = (test_preds_proba >= best_thresh).astype(int)

submission = pd.DataFrame({
    'ID': test_ids,
    'completed': final_test_preds
})
submission.to_csv('submission_lgbm_tuned.csv', index=False)
print("--- 'submission_lgbm_tuned.csv' 파일 생성이 완료되었습니다. ---")

=== 5-Fold LightGBM 학습 시작 ===
Fold 1 학습 완료
Fold 2 학습 완료
Fold 3 학습 완료
Fold 4 학습 완료
Fold 5 학습 완료

[최종 결과]
최적 임계값: 0.14, 전체 검증 F1 Score: 0.4280
--- 'submission_lgbm_tuned.csv' 파일 생성이 완료되었습니다. ---
